In [1]:
import sys
import os

# --- Bước 1: Thêm thư mục gốc của dự án vào Python Path ---

# Lấy đường dẫn thư mục hiện tại của notebook (.../lesson-03/notebook)
current_dir = os.getcwd()

# Đi lùi 2 cấp để đến thư mục gốc của dự án (.../DS201-DL-PRACTICALLESSON)
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Thêm thư mục gốc vào sys.path nếu nó chưa có ở đó
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Đã thêm vào sys.path: {project_root}")


Đã thêm vào sys.path: c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-03


In [2]:
from src.models.lstm import lstm
from src.utils import Vocab

from src.train import train

import torch
import torch.nn as nn
import pandas as pd
from functools import partial
from torch.utils.data import DataLoader

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Session 00

## Session 01

### 1.1. Requirements

> Xây dựng mạng LSTM gồm 5 lớp với hidden size là 256 cho bài toán phân loại văn bản. Huấn luyện mô hình này trên bộ dữ liệu UIT-VSFC (Vietnamese Student Feedback Corpus) sử dụng Adam làm phương thức tối ưu tham số và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 1.2. Configuration

#### 1.2.1. Vocabulary

In [7]:
vocab_path = r'..\dataset\uit_vsvc'
vocab = Vocab(
    vocab_path
)

In [8]:
PAD_IDX = vocab.w2i['<PAD>']

#### 1.2.2. Dataset

In [9]:
from src.utils import VsvcDataset, collate_fn

In [10]:
try:
    _train_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-train.json')
    _dev_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-dev.json')
    _test_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-test.json')
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file train/dev tại '{vocab_path}'")

train_dataset = VsvcDataset(dataframe=_train_vsvc, vocab=vocab)
dev_dataset = VsvcDataset(dataframe=_dev_vsvc, vocab=vocab)
test_dataset = VsvcDataset(dataframe=_test_vsvc, vocab=vocab)

In [11]:
collate_fn_with_padding = partial(collate_fn, pad_idx=PAD_IDX)

# Tạo loader
vsvc_train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_with_padding
)

vsvc_dev_loader = DataLoader(
    dataset=dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding
)

vsvc_test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding 
)

### 1.3. Training

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
lstm_instance = lstm(
    vocab_size=vocab.vocab_size,
    embedding_dim=256,
    hidden_size=256,
    output_size=vocab.n_labels
).to(device)

print(lstm_instance)

lstm(
  (embedding): Embedding(2879, 256, padding_idx=0)
  (lstm): LSTM(256, 256, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)


In [14]:
optimizer = torch.optim.Adam(lstm_instance.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 50
N_LABELS = vocab.n_labels 
MODEL_PATH   = r"..\checkpoints\lstm\lstm_best_model.pth"
HISTORY_PATH = r"..\checkpoints\lstm\training_history.json"

In [15]:
best_model, history_data = train(
    model=lstm_instance,
    train_loader=vsvc_train_loader,
    val_loader=vsvc_dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=NUM_EPOCHS,
    n_labels=N_LABELS,
    model_save_path=MODEL_PATH,
    history_save_path=HISTORY_PATH,
    patience=10
)

--- Bắt đầu training ---
Lưu model tốt nhất tại: ..\checkpoints\lstm\lstm_best_model.pth
Lưu lịch sử training tại: ..\checkpoints\lstm\training_history.json


Epoch 1/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 93.31it/s] 


Epoch 1: Train Loss: 0.8551 | Val Loss: 0.8383 | Val F1: 0.7271
🎉 New best F1: 0.7271. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


Epoch 2/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 141.85it/s]


Epoch 2: Train Loss: 0.7456 | Val Loss: 0.6764 | Val F1: 0.7720
🎉 New best F1: 0.7720. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


...

Epoch 20: Train Loss: 0.0639 | Val Loss: 0.5679 | Val F1: 0.8572
No improvement. Patience: 8/10


Epoch 21/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 151.14it/s]


Epoch 21: Train Loss: 0.0597 | Val Loss: 0.5990 | Val F1: 0.8534
No improvement. Patience: 9/10


Epoch 22/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 153.28it/s]

Epoch 22: Train Loss: 0.0513 | Val Loss: 0.6164 | Val F1: 0.8617
No improvement. Patience: 10/10
Early stopping triggered after 22 epochs.

--- Training finished ---
Best Validation F1-score: 0.8711
Training history successfully saved to ..\checkpoints\lstm\training_history.json


### 1.4. Evaluation

In [16]:
from src.evaluate import evaluate

In [17]:
label_names = [vocab.i2l[i] for i in range(vocab.n_labels)]
print(f"Các nhãn (theo thứ tự): {label_names}")

Các nhãn (theo thứ tự): ['training_program', 'lecturer', 'others', 'facility']


In [18]:
test_results = evaluate(
    model=best_model,
    test_loader=vsvc_test_loader,
    criterion=criterion,
    device=device,
    n_labels=N_LABELS,
    label_names=label_names
)

--- Bắt đầu đánh giá trên tập Test ---


Evaluating:   0%|          | 0/99 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 99/99 [00:00<00:00, 116.09it/s]


--- 🏁 Kết quả Đánh giá trên tập Test ---
Thời gian đánh giá: 0.86 giây
Test Loss: 	0.4019
Test Accuracy: 	86.86%
Test F1-Score (Macro): 	0.7482

📊 Báo cáo chi tiết (Classification Report):
                  precision    recall  f1-score   support

training_program       0.73      0.74      0.73       572
        lecturer       0.92      0.93      0.93      2290
          others       0.48      0.40      0.44       159
        facility       0.87      0.91      0.89       145

        accuracy                           0.87      3166
       macro avg       0.75      0.75      0.75      3166
    weighted avg       0.87      0.87      0.87      3166



## Session 02

### 2.1. Requirements

Xây dựng mạng GRU gồm 5 lớp với hidden size là 256 cho bài toán phân loại văn bản. Huấn luyện mô hình này trên bộ dữ liệu UIT-VSFC (Vietnamese Student Feedback Corpus) sử dụng Adam làm phương thức tối ưu tham số và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 2.2. Configuration

#### 2.2.1. Vocabulary

In [19]:
vocab_path = r'..\dataset\uit_vsvc'
vocab = Vocab(
    vocab_path
)
PAD_IDX = vocab.w2i['<PAD>']

#### 2.2.2. Dataset

In [20]:
from src.utils import VsvcDataset, collate_fn

In [21]:
try:
    _train_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-train.json')
    _dev_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-dev.json')
    _test_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-test.json')
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file train/dev tại '{vocab_path}'")

train_dataset = VsvcDataset(dataframe=_train_vsvc, vocab=vocab)
dev_dataset = VsvcDataset(dataframe=_dev_vsvc, vocab=vocab)
test_dataset = VsvcDataset(dataframe=_test_vsvc, vocab=vocab)

In [22]:
collate_fn_with_padding = partial(collate_fn, pad_idx=PAD_IDX)

# Tạo loader
vsvc_train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_with_padding
)

vsvc_dev_loader = DataLoader(
    dataset=dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding
)

vsvc_test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding 
)

### 2.3. Training

In [23]:
from src.models.gru import gru

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
optimizer = torch.optim.Adam(lstm_instance.parameters(), lr=0.5)
criterion = nn.CrossEntropyLoss()

N_LABELS = vocab.n_labels 
GRU_NUM_EPOCHS = 50
GRU_MODEL_PATH = r'../checkpoints/gru/gru_best_model.pth'
GRU_HISTORY_PATH = r'../checkpoints/gru/training_history.json'

In [42]:
gru_instance = gru(
    vocab_size = vocab.vocab_size, 
    embedding_dim = 256, 
    hidden_size = 256, 
    num_layers = 5,
    num_classes = vocab.n_labels,
    dropout=0.5
).to(device)

print(gru_instance)

gru(
  (embedding): Embedding(2879, 256, padding_idx=0)
  (gru): GRU(256, 256, num_layers=5, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=256, out_features=4, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [43]:
from src.train import train

In [44]:
gru_best_model, gru_history_data = train(
    model=gru_instance,                 #
    train_loader=vsvc_train_loader,     
    val_loader=vsvc_dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=GRU_NUM_EPOCHS,              #
    n_labels=N_LABELS,                      #
    model_save_path=GRU_MODEL_PATH,         #
    history_save_path=GRU_HISTORY_PATH,     #
    patience=10
)

--- Bắt đầu training ---
Lưu model tốt nhất tại: ../checkpoints/gru/gru_best_model.pth
Lưu lịch sử training tại: ../checkpoints/gru/training_history.json


Epoch 1/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 81.31it/s]


Epoch 1: Train Loss: 1.4187 | Val Loss: 1.4187 | Val F1: 0.0442
🎉 New best F1: 0.0442. Model saved to ../checkpoints/gru/gru_best_model.pth


Epoch 2/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 83.22it/s]


Epoch 2: Train Loss: 1.4188 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 1/10


...

Epoch 9: Train Loss: 1.4182 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 8/10


Epoch 10/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 72.24it/s]


Epoch 10: Train Loss: 1.4186 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 9/10


Epoch 11/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 84.55it/s]

Epoch 11: Train Loss: 1.4186 | Val Loss: 1.4187 | Val F1: 0.0442
No improvement. Patience: 10/10
Early stopping triggered after 11 epochs.

--- Training finished ---
Best Validation F1-score: 0.0442
Training history successfully saved to ../checkpoints/gru/training_history.json


## Session 03

### 3.1. Requirements

Xây dựng kiến trúc Encoder-Decoder trong đó BiEncoder gồm 5 lớp LSTM và Decoder gồm 5 lớp LSTM với hidden size là 256 cho bài toán nhận diện thực thể (Name Entity Recognition). Huấn luyện mô hình trên bộ dữ liệu PhoNER và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 3.2. Configuration

In [3]:
from src.models.bilstm import BiLSTM